In [15]:
#1 ทำ preprocessing สำหรับ log (For Training Stage 1)

def add_prefix_token(text): # log data ต้องผ่าน code นี้ก่อน training / inference
    # clean log
    text = text.replace("\t", " ")
    text = text.strip()
    # add token
    if text[0:9] == "timestamp":
        return "[LOG]\n" + text
    else:
        return "[SQL]\n" + text

In [ ]:
#1 ทำ preprocessing สำหรับ log (For Training Stage 2)

def add_prefix_token(text): # log data ต้องผ่าน code นี้ก่อน training / inference
    # clean log
    text = text.replace("\t", " ")
    text = text.strip()
    # add token
    if text[0].isalpha() or text[3].isalpha():
        return "[SQL]\n" + text
    else:
        return "[LOG]\n" + text

In [16]:
#2 โหลด CSV + clean

from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="All Data stage 1.csv"
)["train"]

dataset = dataset.map(
    lambda x: {"text": add_prefix_token(x["query log"])}
)

dataset = dataset.remove_columns(["query log", "status"])
dataset = dataset.rename_column("label", "labels")


In [17]:
#3 ทำ Tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)


In [8]:
tokenizer("hello world")


{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [18]:
#4 Train / Validation Split

dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]


In [19]:
#5 โหลดโมเดลสำหรับ Binary Classification

from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
#6 Training Configuration (เหมาะกับ Log)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="Finetuned Bert Model State 1",
    eval_strategy="epoch", #เลิกใช้ evaluation_strategy แล้ว
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",   # 🔴 ปิด wandb
)


In [23]:
#7 Metric (สำคัญมากสำหรับ Anomaly)

from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [24]:
#8 เริ่ม Fine-tune 🚀

from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]  #หยุด Train เมื่อค่า F1 ไม่ดีขึ้น
)

trainer.train()


C:\Users\aungl\AppData\Local\Temp\ipykernel_26972\3697966789.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.570132,0.717647,0.700000,0.350000,0.466667
2,0.616200,0.528189,0.729412,0.666667,0.466667,0.549020
3,0.474800,0.515457,0.764706,0.651515,0.716667,0.682540
4,0.406900,0.508362,0.764706,0.666667,0.666667,0.666667


TrainOutput(global_step=172, training_loss=0.4825930539951768, metrics={'train_runtime': 538.6686, 'train_samples_per_second': 5.035, 'train_steps_per_second': 0.319, 'total_flos': 713557182136320.0, 'train_loss': 0.4825930539951768, 'epoch': 4.0})